### Univariate Analysis

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from typing import Dict, Any, cast
from matplotlib.patches import Rectangle

def numeric_timeseries_univariate_analysis(s: pl.Series) -> Dict[str, Any]:
    """
    Brutally optimized univariate analysis for a numeric Time Series.
    Performance Architecture:
     - Parallel Execution: Computes full numeric stats (skew, kurtosis, percentiles) 
       in a single parallel pass over the entire dataset.
     - Strided Downsampling: Uses deterministic skipping (gather) rather than random 
       sampling to preserve the true chronological shape without freezing Seaborn.
    """
    warnings.filterwarnings("ignore")
    name = s.name if s.name else "value"
    total_rows = len(s)
    if total_rows == 0:
        raise ValueError("Series is empty.")
        
    # 1. Parallel Statistical Computation (Exact Numeric Stats)
    df = s.to_frame()
    stats = df.select([
        pl.len().alias("total_rows"),
        pl.col(name).null_count().alias("null_count"),
        pl.col(name).n_unique().alias("unique"),
        pl.col(name).mean().alias("mean"),
        pl.col(name).median().alias("median"),
        pl.col(name).min().alias("min"),
        pl.col(name).max().alias("max"),
        pl.col(name).var().alias("var"),
        pl.col(name).std().alias("std"),
        pl.col(name).skew().alias("skew"),
        pl.col(name).kurtosis().alias("kurtosis"),
        pl.col(name).quantile(0.25).alias("p25"),
        pl.col(name).quantile(0.50).alias("p50"),
        pl.col(name).quantile(0.75).alias("p75"),
        pl.col(name).quantile(0.99).alias("p99"),
    ])
    
    stats_dict: Dict[str, Any] = stats.to_dicts()[0]
    null_count = int(stats_dict["null_count"])
    stats_dict["null_pct"] = (null_count / total_rows) * 100
    
    # Mode and Mode Percentage via Polars parallel hash aggregation
    vc = s.value_counts(sort=True)
    if vc.height > 0:
        row_data = vc.row(0)
        mode_val = row_data[0]
        mode_count = int(row_data[1])
        stats_dict["mode"] = mode_val
        stats_dict["mode_pct"] = (mode_count / total_rows) * 100
    else:
        stats_dict["mode"] = None
        stats_dict["mode_pct"] = 0.0
        
    # 2. Smart Strided Downsampling (Chronology Preservation)
    # We append a row index to act as our Time Step (X-axis)
    df_indexed = df.with_row_index(name="time_step")
    df_clean = df_indexed.drop_nulls(subset=[name])
    
    if s.dtype in [pl.Float32, pl.Float64]:
        df_clean = df_clean.filter(pl.col(name).is_not_nan())
        
    clean_len = df_clean.height
    
    # Max out at ~250k points to prevent rendering freezes, keeping chronological order
    step_size = max(1, clean_len // 250_000)
    if step_size > 1 and clean_len > 0:
        indices = pl.int_range(0, clean_len, step=step_size, dtype=pl.UInt32)
        df_plot = df_clean.gather(indices)
    else:
        df_plot = df_clean
        
    time_steps = df_plot.get_column("time_step").to_numpy()
    vals_array = df_plot.get_column(name).to_numpy()
        
    # 3. Modern Business Presentation Plotting
    sns.set_theme(style="whitegrid", palette="muted")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2.5, 1]})
    fig.patch.set_facecolor('#F8F9FA')
    for ax in axes:
        ax.set_facecolor('#F8F9FA')
        
    def safe_fmt(val: Any, decimals: int = 4) -> str:
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return "N/A"
        return f"{val:.{decimals}f}"
        
    if len(vals_array) > 0:
        # Left Plot: Chronological Trend (Index as Time)
        sns.lineplot(x=time_steps, y=vals_array, ax=axes[0], 
                     color='#E63946', linewidth=1.5)
        axes[0].fill_between(time_steps, vals_array, color='#E63946', alpha=0.1)
        
        # Overlay Mean & Median on the time series
        mean_val = stats_dict["mean"]
        median_val = stats_dict["median"]
        axes[0].axhline(mean_val, color='#2A9D8F', linestyle='--', linewidth=2, label=f'Mean: {safe_fmt(mean_val)}')
        axes[0].axhline(median_val, color='#F4A261', linestyle='-.', linewidth=2, label=f'Median: {safe_fmt(median_val)}')
        
        axes[0].set_title(f"Sequential Trend of '{name}'", fontsize=16, fontweight='bold', color='#1D3557', pad=15)
        axes[0].set_xlabel("Time Step (Row Index)", fontsize=13, fontweight='bold', color='#457B9D')
        axes[0].set_ylabel(name, fontsize=13, fontweight='bold', color='#457B9D')
        axes[0].legend(fontsize=11, frameon=True, shadow=True)
        
        # Right Plot: Value Distribution Shape
        sns.histplot(vals_array, kde=True, ax=axes[1], color='#0077B6', 
                     line_kws={'color': '#1D3557', 'linewidth': 2}, bins=50, alpha=0.7)
        axes[1].set_title(f"Distribution of '{name}'", fontsize=16, fontweight='bold', color='#1D3557', pad=15)
        axes[1].set_xlabel(name, fontsize=13, fontweight='bold', color='#457B9D')
        axes[1].set_ylabel("Frequency", fontsize=13, fontweight='bold', color='#457B9D')
    else:
        axes[0].text(0.5, 0.5, 'No valid data to plot', ha='center', va='center', fontsize=14)
        axes[1].text(0.5, 0.5, 'No valid data to plot', ha='center', va='center', fontsize=14)
        
    # Clean up spines for business look
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#CCCCCC')
        ax.spines['bottom'].set_color('#CCCCCC')
        ax.tick_params(colors='#555555', labelsize=10)
        
    plt.tight_layout()
    plt.show()
    
    # 4. Beautiful Console Output
    print(f"\n{'='*55}")
    print(f"  UNIVARIATE ANALYSIS (TIME SERIES): {name.upper()}")
    print(f"{'='*55}")
    print(f"  Total Rows:        {total_rows:>15,}")
    print(f"  Null Count:        {null_count:>15,} ({stats_dict['null_pct']:.2f}%)")
    print(f"  Unique Values:     {stats_dict['unique']:>15,}")
    print(f"  Mean:              {safe_fmt(stats_dict['mean']):>15}")
    print(f"  Median:            {safe_fmt(stats_dict['median']):>15}")
    print(f"  Mode:              {safe_fmt(stats_dict['mode'])} ({stats_dict['mode_pct']:.2f}%)")
    print(f"  Min:               {safe_fmt(stats_dict['min']):>15}")
    print(f"  Max:               {safe_fmt(stats_dict['max']):>15}")
    print(f"  Variance:          {safe_fmt(stats_dict['var']):>15}")
    print(f"  Std Dev:           {safe_fmt(stats_dict['std']):>15}")
    print(f"  Skewness:          {safe_fmt(stats_dict['skew']):>15}")
    print(f"  Kurtosis:          {safe_fmt(stats_dict['kurtosis']):>15}")
    print(f"  25th Percentile:   {safe_fmt(stats_dict['p25']):>15}")
    print(f"  50th Percentile:   {safe_fmt(stats_dict['p50']):>15}")
    print(f"  75th Percentile:   {safe_fmt(stats_dict['p75']):>15}")
    print(f"  99th Percentile:   {safe_fmt(stats_dict['p99']):>15}")
    print(f"{'='*55}\n")
    
    return stats_dict

def categorical_timeseries_univariate_analysis(s: pl.Series) -> Dict[str, Any]:
    """
    Brutally optimized univariate analysis for a Categorical Time Series.
    Performance Architecture:
    - Universal Type Handling: Safely casts any dtype (Int flags like 0/1, Bools, Strings) 
      into a String-based Categorical to guarantee stable physical mapping and 
      prevent Seaborn type errors on binary/class flags.
    - Parallel Execution: Computes categorical stats (frequencies, nulls, unique)
      in a single parallel pass over the entire dataset.
    - Strided Downsampling: Uses deterministic skipping (gather) rather than random
      sampling to preserve the true chronological sequence of category transitions
      without freezing Seaborn.
    - Business Design: Generates a clean, modern, presentation-ready dashboard
      showing category evolution over time alongside Pareto distribution.
    """
    warnings.filterwarnings("ignore")
    name = s.name if s.name else "category"
    total_rows = len(s)
    if total_rows == 0:
        raise ValueError("Series is empty.")

    # 1. Data Validation & Universal Mapping
    # Force any dtype (Int, Bool, Float, String) into a String-based Categorical.
    # This guarantees stable physical integer mapping for the scatter plot and 
    # prevents Seaborn from throwing type errors on flags (0/1) or booleans.
    if s.dtype not in (pl.Categorical, pl.Enum):
        s = s.cast(pl.String).cast(pl.Categorical)

    # 2. Core Metrics (Polars Native - highly parallelized O(N))
    df = s.to_frame()
    stats = df.select([
        pl.len().alias("total_rows"),
        pl.col(name).null_count().alias("null_count"),
        pl.col(name).n_unique().alias("unique"),
    ])
    stats_dict: Dict[str, Any] = stats.to_dicts()[0]
    null_count = int(stats_dict["null_count"])
    stats_dict["null_pct"] = (null_count / total_rows) * 100

    # Frequency Count (Polars Hash Aggregation)
    freq_df = s.value_counts(sort=True)
    if freq_df.height > 0:
        row_data = freq_df.row(0)
        mode_val = row_data[0]
        mode_count = int(row_data[1])
        stats_dict["mode"] = mode_val
        stats_dict["mode_pct"] = (mode_count / total_rows) * 100
    else:
        stats_dict["mode"] = None
        stats_dict["mode_pct"] = 0.0
        
    stats_dict["frequency_count_top_15"] = freq_df.head(15)

    # 3. Smart Strided Downsampling (Chronology Preservation)
    # We append a row index to act as our Time Step (X-axis)
    df_indexed = df.with_row_index(name="time_step")
    df_clean = df_indexed.drop_nulls(subset=[name])
    clean_len = df_clean.height
    
    # Max out at ~250k points to prevent rendering freezes, keeping chronological order
    step_size = max(1, clean_len // 250_000)
    if step_size > 1 and clean_len > 0:
        indices = pl.int_range(0, clean_len, step=step_size, dtype=pl.UInt32)
        df_plot = df_clean.gather(indices)
    else:
        df_plot = df_clean
        
    time_steps = df_plot.get_column("time_step").to_numpy()
    
    # Map to physical codes for efficient scatter plotting
    phys_series_plot = df_plot.get_column(name).to_physical()
    vals_array = phys_series_plot.to_numpy()
    
    # Extract category mapping for Y-axis labels
    cat_mapping = s.cat.get_categories().to_list()
    unique_codes = np.unique(vals_array)
    valid_codes = [int(c) for c in unique_codes if 0 <= c < len(cat_mapping)]

    # 4. Modern Business Presentation Plotting
    sns.set_theme(style="whitegrid", palette="muted")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2.5, 1]})
    fig.patch.set_facecolor('#F8F9FA')
    for ax in axes:
        ax.set_facecolor('#F8F9FA')

    def safe_fmt(val: Any, decimals: int = 4) -> str:
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return "N/A"
        if isinstance(val, float):
            return f"{val:.{decimals}f}"
        return str(val)

    if len(time_steps) > 0 and len(freq_df) > 0:
        # Left Plot: Chronological Trend (Scatter plot of Category over Time)
        sns.scatterplot(x=time_steps, y=vals_array, ax=axes[0], 
                        color='#0077B6', alpha=0.6, s=15, edgecolor='none')
        
        # Map Y-axis ticks back to category names safely
        if len(valid_codes) <= 20:
            axes[0].set_yticks(valid_codes)
            axes[0].set_yticklabels([cat_mapping[c] for c in valid_codes], fontsize=10)
        else:
            step = max(1, len(valid_codes) // 15)
            tick_positions = valid_codes[::step]
            axes[0].set_yticks(tick_positions)
            axes[0].set_yticklabels([cat_mapping[c] for c in tick_positions], fontsize=9, rotation=45)
            
        axes[0].set_title(f"Sequential Category Evolution of '{name}'", fontsize=16, fontweight='bold', color='#1D3557', pad=15)
        axes[0].set_xlabel("Time Step (Row Index)", fontsize=13, fontweight='bold', color='#457B9D')
        axes[0].set_ylabel("Category", fontsize=13, fontweight='bold', color='#457B9D')
        
        # Right Plot: Pareto Analysis (Distribution Shape)
        plot_df = freq_df.head(15).to_pandas()
        cat_col_name = str(s.name) if s.name and str(s.name) in plot_df.columns else str(plot_df.columns[0])
        plot_df.rename(columns={cat_col_name: 'Category', 'count': 'Frequency'}, inplace=True)
        total_valid = df_clean.height
        plot_df['Cumulative_Pct'] = (plot_df['Frequency'].cumsum() / total_valid) * 100
        
        sns.barplot(
            data=plot_df, x='Category', y='Frequency', 
            palette="mako_r", ax=axes[1], edgecolor='white', linewidth=1.2
        )
        axes[1].set_xlabel('Category', fontsize=12, fontweight='bold', color='#333333', labelpad=10)
        axes[1].set_ylabel('Frequency Count', fontsize=12, fontweight='bold', color='#333333', labelpad=10)
        axes[1].set_title(f"Pareto Distribution of '{name}'", fontsize=16, fontweight='bold', color='#1D3557', pad=15)
        axes[1].tick_params(axis='x', rotation=45, labelsize=10, colors='#555555')
        axes[1].tick_params(axis='y', labelsize=10, colors='#555555')
        
        # Data labels on bars
        for p in axes[1].patches:
            rect = cast(Rectangle, p)
            height = rect.get_height()
            if height > 0:
                axes[1].annotate(f'{int(height):,}',
                            (rect.get_x() + rect.get_width() / 2., height),
                            ha='center', va='bottom',
                            fontsize=9, color='#333333', fontweight='bold',
                            xytext=(0, 3), textcoords='offset points')
                                
        # Secondary axis for Cumulative Percentage
        ax2 = axes[1].twinx()
        ax2.set_ylabel('Cumulative Percentage (%)', fontsize=12, fontweight='bold', color='#d62728', labelpad=10)
        ax2.tick_params(axis='y', labelsize=10, colors='#d62728')
        ax2.set_ylim(0, 105)
        
        sns.lineplot(
            data=plot_df, x='Category', y='Cumulative_Pct', 
            ax=ax2, color='#d62728', marker='o', linewidth=2.5, markersize=6
        )
    else:
        axes[0].text(0.5, 0.5, 'No valid data to plot', ha='center', va='center', fontsize=14)
        axes[1].text(0.5, 0.5, 'No valid data to plot', ha='center', va='center', fontsize=14)

    # Clean up spines for business look
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#CCCCCC')
        ax.spines['bottom'].set_color('#CCCCCC')
        ax.tick_params(colors='#555555', labelsize=10)
    
    # Specific cleanup for the secondary axis
    if len(time_steps) > 0 and len(freq_df) > 0:
        ax2.spines['top'].set_visible(False)
        ax2.spines['left'].set_color('#CCCCCC')
        ax2.spines['bottom'].set_color('#CCCCCC')
        ax2.spines['right'].set_color('#d62728')

    plt.tight_layout()
    plt.show()

    # 5. Beautiful Console Output
    print(f"\n{'='*55}")
    print(f"  UNIVARIATE ANALYSIS (CATEGORICAL TIME SERIES): {name.upper()}")
    print(f"{'='*55}")
    print(f"  Total Rows:        {total_rows:>15,}")
    print(f"  Null Count:        {null_count:>15,} ({stats_dict['null_pct']:.2f}%)")
    print(f"  Unique Values:     {stats_dict['unique']:>15,}")
    print(f"  Mode:              {safe_fmt(stats_dict['mode'])} ({stats_dict['mode_pct']:.2f}%)")
    print(f"{'='*55}\n")

    return stats_dict

### Load Datasets

In [ ]:
north_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/north_raw.parquet")
south_df = pl.read_parquet("/kaggle/input/datasets/tuboa2/water-quality-index/south_raw.parquet")

### Get Column Names

In [ ]:
print(north_df.columns)

### Weekends

In [ ]:
print(categorical_timeseries_univariate_analysis(s=north_df["is_weekend"]))
print(categorical_timeseries_univariate_analysis(s=south_df["is_weekend"]))

### Seasons

In [ ]:
print(categorical_timeseries_univariate_analysis(s=north_df["season_label"]))
print(categorical_timeseries_univariate_analysis(s=south_df["season_label"]))

### Daily Max Temperature

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["daily_max_temp_celsius"]))
print(numeric_timeseries_univariate_analysis(s=south_df["daily_max_temp_celsius"]))

### Temperature Anomaly

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["temp_anomaly_celsius"]))
print(numeric_timeseries_univariate_analysis(s=south_df["temp_anomaly_celsius"]))

### Cumulative Heat Index

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cumulative_heat_index"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cumulative_heat_index"]))

### Daily Rainfall

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["daily_rainfall_mm"]))
print(numeric_timeseries_univariate_analysis(s=south_df["daily_rainfall_mm"]))

### Consecutive Dry Days

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["consecutive_dry_days"]))
print(numeric_timeseries_univariate_analysis(s=south_df["consecutive_dry_days"]))

### Rolling Rainfall MM for 7 Days

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["rolling_7d_rainfall_mm"]))
print(numeric_timeseries_univariate_analysis(s=south_df["rolling_7d_rainfall_mm"]))

### Cumulative Storm Rainfall

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cumulative_storm_rainfall_mm"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cumulative_storm_rainfall_mm"]))

### Antecedent Moisture Condition

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["antecedent_moisture_condition"]))
print(numeric_timeseries_univariate_analysis(s=south_df["antecedent_moisture_condition"]))

### Daily Runoff Volume

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["daily_runoff_volume_m3"]))
print(numeric_timeseries_univariate_analysis(s=south_df["daily_runoff_volume_m3"]))

### Total Suspended Solids

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["total_suspended_solids_mg_L"]))
print(numeric_timeseries_univariate_analysis(s=south_df["total_suspended_solids_mg_L"]))

### Nutrient Load Index

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["nutrient_load_index"]))
print(numeric_timeseries_univariate_analysis(s=south_df["nutrient_load_index"]))

### Heat X Nutrient Synergy

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["heat_x_nutrient_synergy"]))
print(numeric_timeseries_univariate_analysis(s=south_df["heat_x_nutrient_synergy"]))

### Cluster Heavy Users Daily Mean

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cluster_heavy_users_daily_mean_liters"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cluster_heavy_users_daily_mean_liters"]))

### Cluster Conservationists Daily Mean

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cluster_conservationists_daily_mean_liters"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cluster_conservationists_daily_mean_liters"]))

### Cluster Outdoor Landscape Daily Mean

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cluster_outdoor_landscape_daily_mean_liters"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cluster_outdoor_landscape_daily_mean_liters"]))

### Cluster Standard Consumers Daily Mean

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["cluster_standard_consumers_daily_mean_liters"]))
print(numeric_timeseries_univariate_analysis(s=south_df["cluster_standard_consumers_daily_mean_liters"]))

### Watering Ban Active

In [ ]:
print(categorical_timeseries_univariate_analysis(s=north_df["watering_ban_active"]))
print(categorical_timeseries_univariate_analysis(s=south_df["watering_ban_active"]))

### Holiday Weekend Flag

In [ ]:
print(categorical_timeseries_univariate_analysis(s=north_df["holiday_weekend_flag"]))
print(categorical_timeseries_univariate_analysis(s=south_df["holiday_weekend_flag"]))

### Tiered Pricing Regime

In [ ]:
print(categorical_timeseries_univariate_analysis(s=north_df["tiered_pricing_regime"]))
print(categorical_timeseries_univariate_analysis(s=south_df["tiered_pricing_regime"]))

### Drought X Heat Stress

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["drought_x_heat_stress"]))
print(numeric_timeseries_univariate_analysis(s=south_df["drought_x_heat_stress"]))

### Demand X Runoff Pressure

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["demand_x_runoff_pressure"]))
print(numeric_timeseries_univariate_analysis(s=south_df["demand_x_runoff_pressure"]))

### Water Quality Index

In [ ]:
print(numeric_timeseries_univariate_analysis(s=north_df["water_quality_index"]))
print(numeric_timeseries_univariate_analysis(s=south_df["water_quality_index"]))